## Install Dependencies

Run this cell to install required packages (uncomment if needed):

In [55]:
# !pip install langchain langchain-openai langchain-google-genai langchain-neo4j python-dotenv

## Import Libraries and Load Environment Variables

In [56]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage

# Load environment variables
load_dotenv(override=True)

# Verify keys are loaded
print("OpenAI Key loaded:", bool(os.getenv("OPENAI_KEY")))
print("OpenRouter Key loaded:", bool(os.getenv("OPEN_ROUTER_KEY")))
print("Google API Key loaded:", bool(os.getenv("GEMINI_KEY")))

OPENAI_KEY = KEY = os.getenv("OPENAI_KEY")
OPENROUTER_KEY = KEY = os.getenv("OPEN_ROUTER_KEY")
GEMINI_KEY = KEY = os.getenv("GEMINI_KEY")

OpenAI Key loaded: True
OpenRouter Key loaded: True
Google API Key loaded: True


## LLM Functions

Functions for each model that take context and question as parameters.

In [57]:
# Persona: Define the assistant's role with comprehensive guidelines
SystemPrompt = """You are an expert Fantasy Premier League (FPL) assistant with deep knowledge of player statistics, team performance, fixtures, and FPL strategy. You provide accurate, data-driven insights to help FPL managers make informed decisions.

Guidelines for your responses:
- Carefully analyze the context provided to extract relevant information
- Answer questions accurately using ONLY the information available in the context
- If the context doesn't contain enough information to answer the question, clearly state that
- Provide specific statistics, player names, team names, and gameweek data when available in the context
- Be concise but comprehensive in your answers
- Use FPL terminology correctly (e.g., GW for gameweek, xG for expected goals, xA for expected assists, ICT index, bonus points, BPS, etc.)
- When discussing player performance, include relevant metrics like points scored, goals, assists, clean sheets, and bonus points if available
- For team-related questions, reference fixtures, form, and statistics from the context
- Maintain an enthusiastic and knowledgeable tone about Fantasy Premier League
- Base all recommendations and insights strictly on the provided context to avoid hallucinations"""

In [58]:
import time

def openai_generate(context: str, question: str) -> dict:
    """
    Generate response using OpenAI model.
    
    Args:
        context: The retrieved knowledge graph information (nodes, relationships, data)
        question: The user's question to answer
        
    Returns:
        Dictionary containing:
        - response: String response from the model
        - metrics: Dictionary with response_time, token_usage, and cost
    """
    openai_llm = ChatOpenAI(
        model="gpt-5.1",
        temperature=0.7,
        api_key=OPENAI_KEY,
        max_tokens=1000
    )
    
    # Structure: Persona (SystemMessage) + Context + Task (HumanMessage)
    messages = [
        SystemMessage(content=SystemPrompt),
        HumanMessage(content=f"""Context:
{context}

Task:
Answer the following question using ONLY the information provided in the context above. If the context doesn't contain enough information to answer the question, clearly state that. Be specific and cite relevant statistics, player names, or data from the context.

Question: {question}""")
    ]
    
    # Measure response time
    start_time = time.time()
    response = openai_llm.invoke(messages)
    end_time = time.time()
    
    # Extract token usage
    prompt_tokens = response.response_metadata.get('token_usage', {}).get('prompt_tokens', 0)
    completion_tokens = response.response_metadata.get('token_usage', {}).get('completion_tokens', 0)
    total_tokens = response.response_metadata.get('token_usage', {}).get('total_tokens', 0)
    
    # Calculate cost (GPT-5.1 pricing: $1.25 per 1M prompt tokens, $10 per 1M completion tokens)
    cost = (prompt_tokens / 1000000 * 1.25) + (completion_tokens / 1000000 * 10)
    
    return {
        "response": response.content,
        "metrics": {
            "response_time": round(end_time - start_time, 2),
            "token_usage": {
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": total_tokens
            },
            "cost": round(cost, 6)
        }
    }

In [59]:
def openrouter_generate(context: str, question: str) -> dict:
    """
    Generate response using OpenRouter model.
    
    Args:
        context: The retrieved knowledge graph information (nodes, relationships, data)
        question: The user's question to answer
        
    Returns:
        Dictionary containing:
        - response: String response from the model
        - metrics: Dictionary with response_time, token_usage, and cost
    """
    openrouter_llm = ChatOpenAI(
        model="meta-llama/llama-3.3-70b-instruct:free",
        temperature=0.7,
        api_key=OPENROUTER_KEY,
        base_url="https://openrouter.ai/api/v1",
        max_tokens=1000,
        default_headers={
            "HTTP-Referer": "http://localhost",
            "X-Title": "LangChain Template"
        }
    )
    
    # Structure: Persona (SystemMessage) + Context + Task (HumanMessage)
    messages = [
        SystemMessage(content=SystemPrompt),
        HumanMessage(content=f"""Context:
{context}

Task:
Answer the following question using ONLY the information provided in the context above. If the context doesn't contain enough information to answer the question, clearly state that. Be specific and cite relevant statistics, player names, or data from the context.

Question: {question}""")
    ]
    
    # Measure response time
    start_time = time.time()
    response = openrouter_llm.invoke(messages)
    end_time = time.time()
    
    # Extract token usage
    prompt_tokens = response.response_metadata.get('token_usage', {}).get('prompt_tokens', 0)
    completion_tokens = response.response_metadata.get('token_usage', {}).get('completion_tokens', 0)
    total_tokens = response.response_metadata.get('token_usage', {}).get('total_tokens', 0)
    
    # Cost for free model is $0
    cost = 0.0
    
    return {
        "response": response.content,
        "metrics": {
            "response_time": round(end_time - start_time, 2),
            "token_usage": {
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": total_tokens
            },
            "cost": round(cost, 6)
        }
    }

In [60]:
def gemini_generate(context: str, question: str) -> dict:
    """
    Generate response using Gemini model.
    
    Args:
        context: The retrieved knowledge graph information (nodes, relationships, data)
        question: The user's question to answer
        
    Returns:
        Dictionary containing:
        - response: String response from the model
        - metrics: Dictionary with response_time, token_usage, and cost
    """
    gemini_llm = ChatGoogleGenerativeAI(
        model="gemini-2.5-flash",
        temperature=0.7,
        google_api_key=GEMINI_KEY,
        max_tokens=1000
    )
    
    # Structure: Persona (SystemMessage) + Context + Task (HumanMessage)
    messages = [
        SystemMessage(content=SystemPrompt),
        HumanMessage(content=f"""Context:
{context}

Task:
Answer the following question using ONLY the information provided in the context above. If the context doesn't contain enough information to answer the question, clearly state that. Be specific and cite relevant statistics, player names, or data from the context.

Question: {question}""")
    ]
    
    # Measure response time
    start_time = time.time()
    response = gemini_llm.invoke(messages)
    end_time = time.time()
    
    # Extract token usage
    usage_metadata = response.usage_metadata
    prompt_tokens = usage_metadata.get('input_tokens', 0)
    completion_tokens = usage_metadata.get('output_tokens', 0)
    total_tokens = usage_metadata.get('total_tokens', 0)
    
    # Calculate cost (Gemini 2.5 Flash pricing: $0.03 per 1M input tokens, $2.50 per 1M output tokens)
    cost = (prompt_tokens / 1000000 * 0.03) + (completion_tokens / 1000000 * 2.50)
    
    return {
        "response": response.content,
        "metrics": {
            "response_time": round(end_time - start_time, 2),
            "token_usage": {
                "prompt_tokens": prompt_tokens,
                "completion_tokens": completion_tokens,
                "total_tokens": total_tokens
            },
            "cost": round(cost, 6)
        }
    }

## Test Functions (Optional)

Test the functions with dummy data.

In [61]:
# Dummy data for testing
test_context = """
Mohamed Salah (Liverpool) - GW15 Performance:
- Points: 13
- Goals: 2
- Assists: 1
- Bonus: 3
- Price: £13.1m
- Ownership: 58.3%
- Form: 8.2

Erling Haaland (Manchester City) - GW15 Performance:
- Points: 2
- Goals: 0
- Assists: 0
- Bonus: 0
- Price: £15.1m
- Ownership: 82.1%
- Form: 6.8
"""

test_question = "Who performed better in GW15, Salah or Haaland?"

In [62]:

print("="*80)
print("=== OpenAI GPT 5.1 ===")
print("="*80)
result = openai_generate(test_context, test_question)
print(f"\nResponse:\n{result['response']}")
print(f"\nMetrics:")
print(f"  Response Time: {result['metrics']['response_time']}s")
print(f"  Tokens - Input: {result['metrics']['token_usage']['prompt_tokens']}, "
      f"Output: {result['metrics']['token_usage']['completion_tokens']}, "
      f"Total: {result['metrics']['token_usage']['total_tokens']}")
print(f"  Cost: ${result['metrics']['cost']}")

=== OpenAI GPT 5.1 ===

Response:
Mohamed Salah performed better than Erling Haaland in GW15 based on the provided data.

- **Mohamed Salah (Liverpool)**  
  - Points: **13**  
  - Goals: **2**  
  - Assists: **1**  
  - Bonus: **3**  
  - Form: **8.2**

- **Erling Haaland (Manchester City)**  
  - Points: **2**  
  - Goals: **0**  
  - Assists: **0**  
  - Bonus: **0**  
  - Form: **6.8**

Salah outscored Haaland by **11 points** (13 vs 2), contributed **2 goals and 1 assist** compared to Haaland’s zero attacking returns, and also earned **3 bonus points** while Haaland earned none. Based on these GW15 stats, **Salah clearly performed better.**

Metrics:
  Response Time: 4.83s
  Tokens - Input: 428, Output: 197, Total: 625
  Cost: $0.002505


In [63]:

print("\n" + "="*80)
print("=== LLama 3.3 (Openrouter) ===")
print("="*80)
result = openrouter_generate(test_context, test_question)
print(f"\nResponse:\n{result['response']}")
print(f"\nMetrics:")
print(f"  Response Time: {result['metrics']['response_time']}s")
print(f"  Tokens - Input: {result['metrics']['token_usage']['prompt_tokens']}, "
      f"Output: {result['metrics']['token_usage']['completion_tokens']}, "
      f"Total: {result['metrics']['token_usage']['total_tokens']}")
print(f"  Cost: ${result['metrics']['cost']}")


=== LLama 3.3 (Openrouter) ===

Response:
According to the context, Mohamed Salah performed better in GW15. He scored 13 points, with 2 goals, 1 assist, and 3 bonus points, whereas Erling Haaland only scored 2 points, with no goals, assists, or bonus points. This significant difference in points scored (13 vs 2) clearly indicates that Salah had a better performance in GW15.

Metrics:
  Response Time: 4.67s
  Tokens - Input: 537, Output: 84, Total: 621
  Cost: $0.0


In [64]:

print("\n" + "="*80)
print("=== Gemini 2.5 Flash ===")
print("="*80)
result = gemini_generate(test_context, test_question)
print(f"\nResponse:\n{result['response']}")
print(f"\nMetrics:")
print(f"  Response Time: {result['metrics']['response_time']}s")
print(f"  Tokens - Input: {result['metrics']['token_usage']['prompt_tokens']}, "
      f"Output: {result['metrics']['token_usage']['completion_tokens']}, "
      f"Total: {result['metrics']['token_usage']['total_tokens']}")
print(f"  Cost: ${result['metrics']['cost']}")


=== Gemini 2.5 Flash ===

Response:
What a fantastic question, FPL managers! Let's dive into the GW15 performances of these two FPL giants using the provided context.

In GW15, **Mohamed Salah** performed significantly better than Erling Haaland.

Here's a breakdown of their GW15 performances:

*   **Mohamed Salah (Liverpool):**
    *   **Points:** 13
    *   **Goals:** 2
    *   **Assists:** 1
    *   **Bonus:** 3

*   **Erling Haaland (Manchester City):**
    *   **Points:** 2
    *   **Goals:** 0
    *   **Assists:** 0
    *   **Bonus:** 0

Salah outscored Haaland by a massive 11 points in GW15, contributing to two goals and an assist, which also earned him the maximum 3 bonus points!

Metrics:
  Response Time: 2.34s
  Tokens - Input: 440, Output: 317, Total: 757
  Cost: $0.000806
